# Hyperparameter search

[General optimization](general-optimization.ipynb) tuned a model's *parameters*.
**Hyperparameters** are the knobs you set *before* training — regularization
strength, tree depth, *k* in k-means. You can't learn them by gradient descent
on the training loss; you search for the values that give the best
**validation** score.

We drive every search with [`hyperopt-rs`](https://crates.io/crates/hyperopt-rs),
an **Optuna-shaped** framework: you write a *define-by-run* objective that calls
`trial.suggest_*` to propose hyperparameters, and you swap the **sampler** to
change strategy — grid, random, or TPE — while the objective stays identical.

```{note}
Below, `val_error(hp)` is a stand-in for *"train a model with this
hyperparameter and measure its cross-validated error"* (from the [Model
Evaluation](../01d-evaluation/cross-validation.ipynb) chapter). We use a cheap
synthetic objective so the *search strategies* — not model fitting — are the
focus. Its minimum sits near `hp = 2.0`.
```

In [ ]:
:dep hyperopt-rs = { version = "0.1.1" }
use hyperopt_rs::prelude::*;

// Lower is better. In practice this would fit a model and return CV error.
fn val_error(hp: f64) -> f64 {
    (hp - 2.0).powi(2) * 0.1 + 0.05 + 0.02 * (hp * 3.0).sin()
}
println!("objective ready (search the range [0, 5]; lower is better)");

## 1. Grid search

The simplest strategy: evaluate every point on a regular grid. Exhaustive and
reproducible, but the cost explodes with the number of hyperparameters (the
"curse of dimensionality"). `GridSampler` enumerates a grid you hand it:

In [ ]:
{
    // 11 grid points across [0, 5]; GridSampler runs exactly grid_size() trials.
    let grid = GridSampler::new().add_float_grid("hp", &(0..11).map(|i| 0.5 * i as f64).collect::<Vec<f64>>());
    let n = grid.grid_size();
    let study = StudyBuilder::new("grid").direction(Direction::Minimize).sampler(grid).build().unwrap();
    study.optimize(|t| { let hp = t.suggest_float("hp", 0.0, 5.0); Ok(val_error(hp)) }, n).unwrap();
    let best = study.best_trial().unwrap().unwrap();
    println!("grid search  : best hp = {}, error = {:.4}  ({} evals)", best.params[0].value, best.value.unwrap(), n);
}

## 2. Random search

Sample the space at random instead of on a grid. Counter-intuitively this
usually **beats** grid search in higher dimensions: with a fixed budget it tries
more distinct values of each individual hyperparameter, rather than wasting
evaluations on redundant grid combinations. Only the **sampler** changes —
`RandomSampler` in place of `GridSampler`; the objective is identical:

In [ ]:
{
    let study = StudyBuilder::new("random").direction(Direction::Minimize).sampler(RandomSampler::seeded(0)).build().unwrap();
    study.optimize(|t| { let hp = t.suggest_float("hp", 0.0, 5.0); Ok(val_error(hp)) }, 15).unwrap();
    let best = study.best_trial().unwrap().unwrap();
    println!("random search: best hp = {}, error = {:.4}  (15 evals)", best.params[0].value, best.value.unwrap());
}

## 3. Bayesian search with TPE

A **Tree-structured Parzen Estimator** is smarter: it models which regions of the
space have produced good scores so far and concentrates new trials there. With a
small budget it homes in on the optimum more reliably than blind random sampling.
Again only the sampler changes — `TpeSampler` (which wraps the
[`tpe`](https://docs.rs/tpe) crate under `hyperopt-rs`'s uniform `Study` API):

In [ ]:
{
    let study = StudyBuilder::new("tpe").direction(Direction::Minimize).sampler(TpeSampler::seeded(0)).build().unwrap();
    study.optimize(|t| { let hp = t.suggest_float("hp", 0.0, 5.0); Ok(val_error(hp)) }, 15).unwrap();
    let best = study.best_trial().unwrap().unwrap();
    println!("TPE search   : best hp = {}, error = {:.4}  (15 evals)", best.params[0].value, best.value.unwrap());
}

Here **TPE** finds the lowest error (landing near `hp = 1.8`) and **grid** nails
its grid optimum at exactly `2.0`; **random**, unlucky on just 15 draws, settled
at `1.3` — the variance it trades for its simplicity. On **higher-dimensional**
objectives grid's cost explodes and becomes infeasible, and the advantage shifts
decisively to random and TPE — with TPE's modelling paying off most when each
evaluation is expensive.

```{note}
**Ecosystem maturity.** Rust's HPO tooling used to be much thinner than Python's
Optuna / Hyperopt / Ray Tune. [`hyperopt-rs`](https://crates.io/crates/hyperopt-rs)
now closes most of that gap: one define-by-run `Study` API, interchangeable
samplers (grid / random / TPE / CMA-ES), pruning / early-stopping, a persistence
layer, and parallel or distributed trial execution. It's newer and single-author,
so re-verify its state on crates.io before depending on version-specific features —
and the focused [`tpe`](https://docs.rs/tpe) crate it builds on is still available
if you only want the one algorithm.
```

Because only the **sampler** changes between the three cells above, swapping
strategy — or adding pruning — is a one-line edit. Next:
[AutoML](../06-automl/automl-classification.ipynb) — which automates the search
over *model choice*, complementary to this chapter's search over one model's
hyperparameters.